# Track training with mlflow

Below is the model training from `01-multiple_regression_local.ipynb`.
While we can specify what to log precisely by calling the associated functions, `autolog()` provides us with a convenient function to log every basic metric, dataset and parameters.
We don't even need to calculate the score ourselves.
To view the results after executing the cells, run `uvx mlflow server --port 5000` from within the `experiments/` directory.

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

import mlflow

In [9]:
def train_model():
    df = pd.read_csv("../data/Student_Performance.csv")
    X = df[["Hours Studied", "Previous Scores", "Sleep Hours", "Sample Question Papers Practiced"]]
    y = df[["Performance Index"]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

    lin_reg = LinearRegression().fit(X_train, y_train)
    y_hat = lin_reg.predict(X_test)

    mse = mean_squared_error(y_test, y_hat)
    r2 = r2_score(y_test, y_hat)
    print(f"MSE: {mse:.3f}")
    print(f"R2 {r2:.3f}")

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    # ax.scatter(y_test, y_hat, label="accuracy")
    ax.scatter(y_hat, y_test, label="accuracy")
    ax.plot([0, 100], [0, 100], '-', color="tab:orange", label="optimum")

    ax.set(xlabel="Prediction", ylabel="Target")
    ax.legend()
    fig.suptitle("LinearRegression")
    plt.close(fig)
    mlflow.log_figure(fig, "lin_reg-plot.png")
    
    return lin_reg

In [10]:
experiment_name = '02-track_with_mlflow'
mlflow.set_experiment(experiment_name)
mlflow.sklearn.autolog()

with mlflow.start_run():
    model = train_model()
    

2026/10/16 14:56:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/joshua/projects/student_performance/.venv/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/10/16 14:56:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/joshua/projects

MSE: 4.196
R2 0.989
